In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [2]:
COLORS = {
    "primary": "#00995D",
    "primary_light": "#B6D44C",
    "deep": "#004B52",
    "alert": "#F47920",

    "support_rose": "#C59A8B",
    "support_blush": "#E5C6C0",
    "support_sand": "#D5CEC2",
    "support_warm": "#F4E2B1",
    "support_mint": "#C1D0B9",
    "support_ice": "#C7DEE2",

    "danger": "#C94F4F",
    "danger_dark": "#8F2D2D",
    "warning": "#F0B24A",
    "success": "#00995D",
    "info": "#2E7D8A",

    "bg": "#F4F8F6",
    "surface": "#FFFFFF",
    "surface_soft": "#EEF5F1",
    "surface_deep": "#073B3A",
    "border": "#D8E4DD",
    "text": "#18302B",
    "muted": "#6B7D76",
    "grid": "#E8EFEB",
    "white": "#FFFFFF",
}

DIAS_PT = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]


In [3]:
def format_int(v):
    return f"{int(v):,}".replace(",", ".")


def format_money(v):
    return f"R$ {v:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")


def format_pct(v, nd=1):
    return f"{v:.{nd}f}%".replace(".", ",")


def truncate_text(text, n=30):
    text = str(text)
    return text if len(text) <= n else text[: n - 1] + "…"


def plot_layout(title=None, height=380, legend="default", **kwargs):
    base = dict(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(255,255,255,0.76)",
        font=dict(family="Inter, sans-serif", size=12, color=COLORS["text"]),
        margin=dict(l=8, r=8, t=50, b=8),
        height=height,
        hoverlabel=dict(
            bgcolor=COLORS["deep"],
            font_color=COLORS["white"],
            font_size=12,
            font_family="Inter, sans-serif",
        ),
    )

    if title is not None:
        base["title"] = dict(
            text=title,
            x=0,
            font=dict(size=15, color=COLORS["deep"])
        )

    if legend == "default":
        base["legend"] = dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
            bgcolor="rgba(0,0,0,0)",
        )
    elif legend is not None:
        base["legend"] = legend

    base.update(kwargs)
    return base


def color_scale(value, vmin=0, vmax=100):
    import matplotlib.colors as mcolors
    import matplotlib as mpl

    cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "unimed_scale",
        [COLORS["primary"], COLORS["primary_light"], COLORS["alert"], COLORS["danger_dark"]],
    )
    norm = mcolors.Normalize(vmin=vmin, vmax=max(vmax, vmin + 1e-9))
    rgba = cmap(norm(value))
    return mcolors.to_hex(rgba)


def color_scale_list(values, vmin=0, vmax=100):
    values = list(values)
    if len(values) == 0:
        return []
    vmax = vmax if vmax is not None else max(values)
    return [color_scale(v, vmin=vmin, vmax=vmax) for v in values]


In [5]:
def carregar_dados(path="dados_lab_hu.xlsx"):
    df = pd.read_excel(path, sheet_name="data")
    dim = pd.read_excel(path, sheet_name="dim_exames")

    for col in ["Interpretação", "Descrição Exame", "Setor Solicitante", "Unidade"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()

    dim["Descrição Exame"] = dim["Descrição Exame"].astype(str).str.strip().str.upper()

    df["DataHoraPedido"] = pd.to_datetime(df["DataHoraPedido"], errors="coerce")
    df = df.dropna(subset=["DataHoraPedido"]).copy()

    df["Data"] = df["DataHoraPedido"].dt.date
    df["Hora"] = df["DataHoraPedido"].dt.hour
    df["DiaSemana"] = df["DataHoraPedido"].dt.dayofweek
    df["MesAno"] = df["DataHoraPedido"].dt.strftime("%m/%Y")
    df["Turno"] = pd.cut(
        df["Hora"],
        bins=[-1, 6, 12, 18, 23],
        labels=["Madrugada", "Manhã", "Tarde", "Noite"],
    )

    custo_map = dict(zip(dim["Descrição Exame"], dim["CUSTO_EXAME"]))
    intervalo_map = dict(zip(dim["Descrição Exame"], dim["INTERVALOS_CLINICOS"]))

    df["Custo_Unit"] = df["Descrição Exame"].map(custo_map).fillna(3.50)
    df["Intervalo_Clinico_h"] = df["Descrição Exame"].map(intervalo_map).fillna(24)

    df["Flag_Normal"] = (df["Interpretação"] == "NORMAL").astype(int)
    df["Flag_Alterado"] = (df["Interpretação"] != "NORMAL").astype(int)

    df = df.sort_values(["Atendimento", "Descrição Exame", "DataHoraPedido"]).copy()
    grp = df.groupby(["Atendimento", "Descrição Exame"], dropna=False)

    df["Interp_Anterior"] = grp["Interpretação"].shift(1)
    df["DataHora_Anterior"] = grp["DataHoraPedido"].shift(1)
    df["Horas_Desde_Anterior"] = (
        (df["DataHoraPedido"] - df["DataHora_Anterior"]).dt.total_seconds() / 3600
    ).round(1)

    df["Flag_Rep"] = (
        (df["Interpretação"] == "NORMAL") &
        (df["Interp_Anterior"] == "NORMAL")
    ).astype(int)

    df["Flag_Rep_Crit"] = (
        (df["Flag_Rep"] == 1) &
        (df["Horas_Desde_Anterior"] < df["Intervalo_Clinico_h"])
    ).astype(int)

    df["Flag_Rep_Alerta"] = ((df["Flag_Rep"] == 1) & (df["Flag_Rep_Crit"] == 0)).astype(int)
    df["Custo_Rep"] = df["Custo_Unit"] * df["Flag_Rep"]

    return df

In [9]:
df_raw = carregar_dados("dados_lab_hu.xlsx")
df_raw.shape

(49742, 24)

In [10]:
def aplicar_filtros(df_raw, unidade=None, setor=None, data_inicio=None, data_fim=None):
    df = df_raw.copy()

    if unidade:
        df = df[df["Unidade"] == unidade].copy()

    if setor:
        df = df[df["Setor Solicitante"] == setor].copy()

    if data_inicio is not None:
        df = df[df["DataHoraPedido"] >= pd.to_datetime(data_inicio)].copy()

    if data_fim is not None:
        data_fim_ts = pd.to_datetime(data_fim) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        df = df[df["DataHoraPedido"] <= data_fim_ts].copy()

    return df

In [11]:
df = aplicar_filtros(
    df_raw,
    unidade=None,
    setor=None,
    data_inicio=df_raw["DataHoraPedido"].min().date(),
    data_fim=df_raw["DataHoraPedido"].max().date()
)

In [13]:
# CÉLULA AUXILIAR — KPIs e bases gerais

total_exames = len(df)
total_pacientes = df["Atendimento"].nunique()
total_pedidos = df["Pedido"].nunique() if "Pedido" in df.columns else np.nan

normais = int(df["Flag_Normal"].sum())
alterados = total_exames - normais
reps = int(df["Flag_Rep"].sum())
reps_crit = int(df["Flag_Rep_Crit"].sum())
reps_alerta = int(df["Flag_Rep_Alerta"].sum())

taxa_normal = (normais / total_exames * 100) if total_exames else 0
taxa_rep = (reps / total_exames * 100) if total_exames else 0

custo_rep_mes = float(df["Custo_Rep"].sum())
custo_rep_ano = custo_rep_mes * 12

periodo_inicio = pd.to_datetime(df["DataHoraPedido"]).min()
periodo_fim = pd.to_datetime(df["DataHoraPedido"]).max()

print("Total exames:", total_exames)
print("Pacientes:", total_pacientes)
print("Repetições:", reps)
print("Custo repetição:", format_money(custo_rep_mes))

Total exames: 49742
Pacientes: 2540
Repetições: 13982
Custo repetição: R$ 35.547


In [33]:
# CÉLULA 9 — Gráfico: Normalidade por exame

exame_norm = (
    df.groupby("Descrição Exame")
    .agg(Total=("Flag_Normal", "count"), Normais=("Flag_Normal", "sum"))
    .reset_index()
)

exame_norm = exame_norm[exame_norm["Total"] >= 5].copy()
exame_norm["Pct_Normal"] = (exame_norm["Normais"] / exame_norm["Total"] * 100).round(1)
exame_norm = exame_norm.sort_values("Pct_Normal", ascending=True).tail(10)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=[truncate_text(x.title(), 34) for x in exame_norm["Descrição Exame"]],
        x=exame_norm["Pct_Normal"],
        orientation="h",
        marker=dict(
            color=color_scale_list(exame_norm["Pct_Normal"], vmin=0, vmax=100),
            line=dict(color="rgba(255,255,255,0.65)", width=1),
        ),
        text=[f"{v:.0f}%" for v in exame_norm["Pct_Normal"]],
        textposition="outside",
        customdata=exame_norm["Total"],
        hovertemplate="<b>%{y}</b><br>Normalidade: %{x:.1f}%<br>Volume: %{customdata:,}<extra></extra>",
    )
)

fig.add_vline(
    x=75,
    line_dash="dot",
    line_color=COLORS["deep"],
    opacity=0.5,
    annotation_text="referência visual 75%",
    annotation_position="top",
)

fig.update_layout(
    **plot_layout("Normalidade por exame", height=430),
    xaxis=dict(title=None, range=[0, 110], ticksuffix="%", showgrid=True, gridcolor=COLORS["grid"]),
    yaxis=dict(title=None, showgrid=False),
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [30]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\DataSigma\AppData\Local\Programs\Python\Python312\python.exe
3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]


In [31]:
import sys
!{sys.executable} -m pip install -U nbformat

In [32]:
import nbformat
print(nbformat.__version__)

5.10.4
